In [89]:
import os
import pandas as pd
import numpy as np

from scipy.stats import pearsonr, ttest_1samp
from statsmodels.stats.multitest import multipletests

# Dataset directory
DATASET_DIR = Path("/Users/nedamohseni/Downloads/SoLo_dataset")

In [90]:
label_cols = [  "feel_lonely", "feel_isolated", "feel_connected"]

all_daily_ema = []

for participant in sorted(os.listdir(DATASET_DIR)):

    if not participant.startswith("pers"):
        continue

    ema_path = os.path.join( DATASET_DIR, participant,  "Self_Report",  "ema_daily.csv")

    if not os.path.exists(ema_path):
        print(f"No EMA file: {participant}")
        continue

    df = pd.read_csv(ema_path)

    # Convert Unix timestamp (ms) -> America/Los_Angeles local time
    df["datetime_la"] = (pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles"))

    # Local calendar date
    df["date"] = df["datetime_la"].dt.date

    # Daily mean of the three EMA labels
    daily = (df.groupby("date", as_index=False)[label_cols].mean())

    daily.insert(0, "participant", participant)

    all_daily_ema.append(daily)

daily_ema = pd.concat(all_daily_ema, ignore_index=True)

print("Participants:", daily_ema["participant"].nunique())
print("Daily rows:", len(daily_ema))

display(daily_ema)

Participants: 31
Daily rows: 3777


,participant,date,feel_lonely,feel_isolated,feel_connected
0,pers2001,2022-01-24,34.333333,30.666667,50.333333
1,pers2001,2022-01-25,0.000000,0.000000,50.000000
2,pers2001,2022-01-26,0.666667,0.000000,98.333333
3,pers2001,2022-01-27,1.000000,0.000000,99.000000
4,pers2001,2022-01-28,1.000000,0.000000,66.000000
...,...,...,...,...,...
3772,pers2038,2022-06-25,16.000000,13.000000,84.000000
3773,pers2038,2022-06-26,19.000000,10.000000,83.000000
3774,pers2038,2022-06-27,5.000000,15.000000,89.000000
3775,pers2038,2022-06-28,9.000000,9.000000,98.000000


In [91]:
all_mobility = []
for participant in sorted(os.listdir(DATASET_DIR)):

    if not participant.startswith("pers"):
        continue

    mobility_path = os.path.join( DATASET_DIR, participant, "AWARE","mobility_features.csv")

    if not os.path.exists(mobility_path):
        print(f"No mobility file: {participant}")
        continue

    df = pd.read_csv(mobility_path)

    df["date"] = pd.to_datetime(df["date"]).dt.date

    all_mobility.append(df)

daily_mobility = pd.concat(all_mobility, ignore_index=True)

print("Participants:", daily_mobility["participant"].nunique())
print("Daily mobility rows:", len(daily_mobility))

display(daily_mobility)

No mobility file: pers2023
No mobility file: pers2033
Participants: 29
Daily mobility rows: 2740


,participant,date,locationvariance,loglocationvariance,totaldistance,avgspeed,varspeed,numberofsignificantplaces,numberlocationtransitions,radiusgyration,...,outlierstimepercent,maxlengthstayatclusters,minlengthstayatclusters,avglengthstayatclusters,stdlengthstayatclusters,locationentropy,normalizedlocationentropy,timeathome,homelabel,minutesdataused
0,pers2001,2021-11-12,7.106724e-06,-5.148331,3098.653878,4.082839,0.771587,1.0,0.0,0.000000,...,0.0,28.470433,28.470433,28.470433,0.000000,0.000000,0.000000,0.000000,1.0,74.007183
1,pers2001,2021-11-13,9.183273e-06,-5.037002,1224.087093,1.906240,22.471702,1.0,0.0,0.000000,...,0.0,29.492517,29.492517,29.492517,0.000000,0.000000,0.000000,17.489150,1.0,68.021367
2,pers2001,2021-11-14,6.354017e-07,-6.196952,48.166381,23.920399,245.344094,1.0,0.0,0.000000,...,0.0,77.395100,77.395100,77.395100,0.000000,0.000000,0.000000,49.142183,1.0,77.515917
3,pers2001,2021-11-15,4.513150e-06,-5.345520,655.108885,1.582665,0.276840,1.0,0.0,0.000000,...,0.0,29.433450,29.433450,29.433450,0.000000,0.000000,0.000000,19.739117,1.0,54.269117
4,pers2001,2021-11-16,9.623346e-06,-5.016674,4656.349609,4.627509,16.689975,3.0,2.0,280.736256,...,0.0,21.790350,1.297767,14.896161,11.776938,0.804399,0.268133,21.790350,1.0,105.062433
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2735,pers2038,2022-06-23,3.046894e-06,-5.516143,2478.181152,3.300678,0.309818,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.048583
2736,pers2038,2022-06-24,9.764165e-06,-5.010365,2642.654719,3.088833,1.358753,2.0,1.0,177.176525,...,0.0,24.266367,1.270700,12.768533,16.260392,0.197805,0.098903,1.270700,1.0,76.870133
2737,pers2038,2022-06-25,3.912699e-05,-4.407524,7738.539029,3.298933,2.039702,1.0,0.0,0.000000,...,0.0,15.416250,15.416250,15.416250,0.000000,0.000000,0.000000,0.000000,1.0,156.162467
2738,pers2038,2022-06-27,2.039813e-05,-4.690410,3069.495385,2.360203,1.108965,1.0,0.0,0.000000,...,0.0,28.369850,28.369850,28.369850,0.000000,0.000000,0.000000,28.369850,1.0,106.401150


In [92]:
merged = daily_mobility.merge(daily_ema, on=["participant", "date"], how="inner")

print("Merged rows:", len(merged))
print("Participants:", merged["participant"].nunique())
display(merged.head())

Merged rows: 1862
Participants: 29


,participant,date,locationvariance,loglocationvariance,totaldistance,avgspeed,varspeed,numberofsignificantplaces,numberlocationtransitions,radiusgyration,...,avglengthstayatclusters,stdlengthstayatclusters,locationentropy,normalizedlocationentropy,timeathome,homelabel,minutesdataused,feel_lonely,feel_isolated,feel_connected
0,pers2001,2022-01-30,3.581604e-07,-6.445922,250.751521,1.827069,0.069280,1.0,0.0,0.0,...,89.391800,0.0,0.0,0.0,89.391800,1.0,97.626350,0.666667,1.333333,32.0
1,pers2001,2022-02-02,4.284296e-06,-5.368121,1092.894346,4.464509,1.436016,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,14.687767,0.000000,3.000000,97.0
2,pers2001,2022-02-11,2.077151e-28,-27.682532,16.534166,1.652499,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.600333,0.000000,0.000000,100.0
3,pers2001,2022-02-12,0.000000e+00,NaN,0.000000,0.000000,0.000000,1.0,0.0,0.0,...,19.506433,0.0,0.0,0.0,19.506433,1.0,19.506433,24.333333,20.666667,0.0
4,pers2001,2022-02-16,0.000000e+00,NaN,30.642937,12.342199,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.148967,0.000000,1.000000,50.0


In [97]:
mobility_features = ["locationvariance", "totaldistance", "varspeed", "radiusgyration", "timeattop1location", "avglengthstayatclusters"]

participant_corrs = []
for feature in mobility_features:
    for label in label_cols:
        for participant, df_p in merged.groupby("participant"):
            pair = df_p[[feature, label]].dropna()
            if len(pair) < 3 or pair[feature].nunique() < 2 or pair[label].nunique() < 2:
                continue
            r, _ = pearsonr(pair[feature], pair[label])
            z = np.arctanh(np.clip(r, -0.999999, 0.999999))
            participant_corrs.append({"participant": participant, "mobility_feature": feature, "ema_label": label, "n_days": len(pair), "r": r, "z": z})

participant_corrs = pd.DataFrame(participant_corrs)

group_results = []

for (feature, label), df_pair in participant_corrs.groupby(["mobility_feature", "ema_label"]):
    z_values = df_pair["z"].dropna().values
    if len(z_values) < 2:
        continue
    mean_z = np.mean(z_values)
    mean_r = np.tanh(mean_z)
    t_stat, p_raw = ttest_1samp(z_values, popmean=0, alternative="two-sided")
    group_results.append({"mobility_feature": feature, "ema_label": label, "n_participants": len(z_values), "mean_r": mean_r, "mean_z": mean_z, "t_stat": t_stat, "p_raw": p_raw})

mobility_results = pd.DataFrame(group_results)
mobility_results["p_fdr"] = multipletests(mobility_results["p_raw"].values, alpha=0.05, method="fdr_bh")[1]
mobility_results["sig_fdr"] = mobility_results["p_fdr"].apply(lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")
mobility_results["table_value"] = mobility_results.apply(lambda row: f"{row['mean_r']:.2f}{row['sig_fdr']}", axis=1)

final_table = mobility_results.pivot(index="mobility_feature", columns="ema_label", values="table_value")
final_table = final_table.reindex(mobility_features)
final_table = final_table[["feel_lonely", "feel_isolated", "feel_connected"]]

final_table


ema_label,feel_lonely,feel_isolated,feel_connected
mobility_feature,,,
locationvariance,-0.10**,-0.08*,0.07
totaldistance,-0.08*,-0.06,0.10*
varspeed,-0.12***,-0.09*,0.10*
radiusgyration,-0.09*,-0.07*,0.08
timeattop1location,0.07,0.14,-0.11*
avglengthstayatclusters,0.09*,0.18,-0.14**


In [101]:
import pandas as pd
import pingouin as pg
from statsmodels.stats.multitest import multipletests

mobility_features = ["locationvariance", "totaldistance", "varspeed", "radiusgyration", "timeattop1location", "avglengthstayatclusters"]

rmcorr_results = []

for feature in mobility_features:
    for label in label_cols:
        pair = merged[["participant", feature, label]].dropna()
        counts = pair.groupby("participant").size()
        valid_participants = counts[counts >= 2].index
        pair = pair[pair["participant"].isin(valid_participants)]

        if pair["participant"].nunique() < 2 or pair[feature].nunique() < 2 or pair[label].nunique() < 2:
            continue

        try:
            result = pg.rm_corr(data=pair, x=feature, y=label, subject="participant")
            ci_col = next((c for c in result.columns if "CI" in c.upper()), None)
            ci_value = result.iloc[0][ci_col] if ci_col is not None else None

            rmcorr_results.append({"mobility_feature": feature, "ema_label": label, "n_participants": pair["participant"].nunique(), "n_observations": len(pair), "r": result.iloc[0]["r"], "dof": result.iloc[0]["dof"], "ci95": ci_value, "p_raw": result.iloc[0]["pval"]})

        except Exception as e:
            print(f"Failed: {feature} × {label}: {repr(e)}")

rmcorr_results = pd.DataFrame(rmcorr_results)

if len(rmcorr_results) > 0:
    rmcorr_results["p_fdr"] = multipletests(rmcorr_results["p_raw"].values, alpha=0.05, method="fdr_bh")[1]
    rmcorr_results["sig_fdr"] = rmcorr_results["p_fdr"].apply(lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")
    rmcorr_results["table_value"] = rmcorr_results.apply(lambda row: f"{row['r']:.2f}{row['sig_fdr']}", axis=1)

    final_table_rm = rmcorr_results.pivot(index="mobility_feature", columns="ema_label", values="table_value")
    final_table_rm = final_table_rm.reindex(mobility_features)
    final_table_rm = final_table_rm[["feel_lonely", "feel_isolated", "feel_connected"]]

    display(final_table_rm)
    display(rmcorr_results)

else:
    print("No repeated-measures correlations were successfully computed.")

ema_label,feel_lonely,feel_isolated,feel_connected
mobility_feature,,,
locationvariance,-0.04,-0.04,0.04
totaldistance,-0.07**,-0.05,0.09***
varspeed,-0.08***,-0.03,0.10***
radiusgyration,-0.05*,-0.05,0.06*
timeattop1location,0.06*,0.04,-0.07**
avglengthstayatclusters,0.13***,0.18***,-0.13***


,mobility_feature,ema_label,n_participants,n_observations,r,dof,ci95,p_raw,p_fdr,sig_fdr,table_value
0,locationvariance,feel_lonely,29,1818,-0.036995,1788,"[-0.08, 0.01]",1.176632e-01,1.363623e-01,,-0.04
1,locationvariance,feel_isolated,29,1815,-0.037468,1785,"[-0.08, 0.01]",1.133450e-01,1.363623e-01,,-0.04
2,locationvariance,feel_connected,29,1786,0.036974,1756,"[-0.01, 0.08]",1.212109e-01,1.363623e-01,,0.04
3,totaldistance,feel_lonely,29,1818,-0.068770,1788,"[-0.11, -0.02]",3.603325e-03,9.265692e-03,**,-0.07**
4,totaldistance,feel_isolated,29,1815,-0.047697,1785,"[-0.09, -0.0]",4.379662e-02,6.064147e-02,,-0.05
5,totaldistance,feel_connected,29,1786,0.090537,1756,"[0.04, 0.14]",1.440112e-04,5.184403e-04,***,0.09***
6,varspeed,feel_lonely,29,1818,-0.084797,1788,"[-0.13, -0.04]",3.286873e-04,9.860620e-04,***,-0.08***
7,varspeed,feel_isolated,29,1815,-0.029360,1785,"[-0.08, 0.02]",2.147807e-01,2.147807e-01,,-0.03
8,varspeed,feel_connected,29,1786,0.098415,1756,"[0.05, 0.14]",3.572878e-05,1.607795e-04,***,0.10***
9,radiusgyration,feel_lonely,29,1660,-0.054561,1630,"[-0.1, -0.01]",2.751698e-02,4.502779e-02,*,-0.05*


In [95]:
print("hi")

hi
